### Human Action Detection

Importing libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from sklearn.utils import resample
#resample is used when we have imbalanced data(walknig 5k and falling 200) and we want to balance it by undersampling or oversampling 
#Oversample Copies minority rows until balanced(falling 5k and walking 5k )
#Undersample Deletes majority rows until balanced(falling 200 and walking 200)

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,GridSearchCV
#what grid search does is it returns the best hyperparameters for model by trying all the combinations 
#hyperparameters are the parameters that we set before training model like trees and depth for random forest learning rate etc
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.preprocessing import RobustScaler

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


In [5]:
df = pd.read_csv(r"C:\Users\sahil\Downloads\Human-Action-Detection\mhealth_raw_data.csv")


In [6]:
df

,alx,aly,alz,glx,gly,glz,arx,ary,arz,grx,gry,grz,Activity,subject
0,2.1849,-9.6967,0.63077,0.103900,-0.84053,-0.68762,-8.6499,-4.5781,0.187760,-0.449020,-1.01030,0.034483,0,subject1
1,2.3876,-9.5080,0.68389,0.085343,-0.83865,-0.68369,-8.6275,-4.3198,0.023595,-0.449020,-1.01030,0.034483,0,subject1
2,2.4086,-9.5674,0.68113,0.085343,-0.83865,-0.68369,-8.5055,-4.2772,0.275720,-0.449020,-1.01030,0.034483,0,subject1
3,2.1814,-9.4301,0.55031,0.085343,-0.83865,-0.68369,-8.6279,-4.3163,0.367520,-0.456860,-1.00820,0.025862,0,subject1
4,2.4173,-9.3889,0.71098,0.085343,-0.83865,-0.68369,-8.7008,-4.1459,0.407290,-0.456860,-1.00820,0.025862,0,subject1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215740,1.7849,-9.8287,0.29725,-0.341370,-0.90056,-0.61493,-3.7198,-8.9071,0.294230,0.041176,-0.99384,-0.480600,0,subject10
1215741,1.8687,-9.8766,0.46236,-0.341370,-0.90056,-0.61493,-3.7160,-8.7455,0.448140,0.041176,-0.99384,-0.480600,0,subject10
1215742,1.6928,-9.9290,0.16631,-0.341370,-0.90056,-0.61493,-3.8824,-9.1155,0.450480,0.041176,-0.99384,-0.480600,0,subject10
1215743,1.5279,-9.6306,0.30458,-0.341370,-0.90056,-0.61493,-3.5564,-9.1441,0.594880,0.041176,-0.99384,-0.480600,0,subject10


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215745 entries, 0 to 1215744
Data columns (total 14 columns):
 #   Column    Non-Null Count    Dtype  
---  ------    --------------    -----  
 0   alx       1215745 non-null  float64
 1   aly       1215745 non-null  float64
 2   alz       1215745 non-null  float64
 3   glx       1215745 non-null  float64
 4   gly       1215745 non-null  float64
 5   glz       1215745 non-null  float64
 6   arx       1215745 non-null  float64
 7   ary       1215745 non-null  float64
 8   arz       1215745 non-null  float64
 9   grx       1215745 non-null  float64
 10  gry       1215745 non-null  float64
 11  grz       1215745 non-null  float64
 12  Activity  1215745 non-null  int64  
 13  subject   1215745 non-null  object 
dtypes: float64(12), int64(1), object(1)
memory usage: 129.9+ MB


In [8]:
df.isnull().sum()

alx         0
aly         0
alz         0
glx         0
gly         0
glz         0
arx         0
ary         0
arz         0
grx         0
gry         0
grz         0
Activity    0
subject     0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
print(df['Activity'].value_counts())
# alot of imbalance is there


Activity
0     872550
1      30720
2      30720
3      30720
4      30720
9      30720
5      30720
11     30720
10     30720
7      29441
8      29337
6      28315
12     10342
Name: count, dtype: int64


In [11]:
data_activity_0=df[df['Activity']==0]
data_activity_else=df[df['Activity']!=0]
data_activity_0=data_activity_0.sample(n=40000)
df=pd.concat([data_activity_0,data_activity_else])


In [12]:
print(df['Activity'].value_counts())



Activity
0     40000
1     30720
2     30720
3     30720
4     30720
9     30720
5     30720
11    30720
10    30720
7     29441
8     29337
6     28315
12    10342
Name: count, dtype: int64


In [13]:
activity_label={
0: "None",
1: "Standing still (1 min)",
2: "Sitting and relaxing (1 min)",
3: "Lying down (1 min)",
4: "Walking (1 min)",
5: "Climbing stairs (1 min)",
6: "Waist bends forward (20x)",
7: "Frontal elevation of arms (20x)",
8: "Knees bending (crouching) (20x)",
9: "Cycling (1 min)",
10: "Jogging (1 min)",
11: "Running (1 min)",
12: "Jump front & back (20x)"
}

In [14]:
# subject1 = df[df['subject'] == 'subject1']
# readings = ['a', 'g']

# for i in range(1, 13):
#     for r in readings:
#         print(f"====================={activity_label[i]} - {r}=====================")
#         plt.figure(figsize=(14, 4))
#         plt.subplot(1, 2, 1)
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "lx"], color='blue', alpha=0.5, label=r + "lx")
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "ly"], color='red', alpha=0.5, label=r + "ly")
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "lz"], color='green', alpha=0.5, label=r + "lz")
#         plt.title("Left ankle sensor")
#         plt.legend()
#         plt.subplot(1, 2, 2)
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "rx"], color='blue', alpha=0.5, label=r + "rx")
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "ry"], color='red', alpha=0.5, label=r + "ry")
#         plt.plot(subject1[subject1['Activity'] == i].reset_index(drop=True)[r + "rz"], color='green', alpha=0.5, label=r + "rz")
#         plt.title("Right ankle sensor")
#         plt.legend()
#         plt.show()

In [15]:
df['Activity'] = df['Activity'].replace(
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    ['None',
     'Standing still (1 min)',
     'Sitting and relaxing (1 min)',
     'Lying down (1 min)',
     'Walking (1 min)',
     'Climbing stairs (1 min)',
     'Waist bends forward (20x)',
     'Frontal elevation of arms (20x)',
     'Knees bending (crouching) (20x)',
     'Cycling (1 min)',
     'Jogging (1 min)',
     'Running (1 min)',
     'Jump front & back (20x)']
)

print(df['Activity'].value_counts())

Activity
None                               40000
Standing still (1 min)             30720
Sitting and relaxing (1 min)       30720
Lying down (1 min)                 30720
Walking (1 min)                    30720
Cycling (1 min)                    30720
Climbing stairs (1 min)            30720
Running (1 min)                    30720
Jogging (1 min)                    30720
Frontal elevation of arms (20x)    29441
Knees bending (crouching) (20x)    29337
Waist bends forward (20x)          28315
Jump front & back (20x)            10342
Name: count, dtype: int64


In [16]:
df1 = df.copy()
#we are doing this becuase if all values is between -1 and 1 and suddenly 1 value is 100 then even standard scaled wont work like mean will be bad
for feature in df1.columns[:-2]: 
    lower_range = np.quantile(df[feature], 0.01)#quantile is basically keeping the values between 0% and 1% called outliner in ascending manner
    upper_range = np.quantile(df[feature], 0.99)
    print(feature, "range:", lower_range, 'to', upper_range)
    
    df1 = df1.drop(df1[(df1[feature] > upper_range) | (df1[feature] < lower_range)].index, axis=0)#so we are keeping the values between 1% and 99% and removing the outliners
    print('shape', df1.shape)

alx range: -11.50542 to 19.223
shape (375542, 14)
aly range: -19.378 to 2.3827119999999993
shape (369628, 14)
alz range: -18.949 to 14.088179999999992
shape (365825, 14)
glx range: -0.75325 to 0.8071615999999957
shape (358772, 14)
gly range: -1.0694 to 0.96435
shape (351951, 14)
glz range: -1.1061 to 0.8290799999999999
shape (346250, 14)
arx range: -21.487 to 9.003905999999999
shape (341021, 14)
ary range: -18.691 to 11.84229999999999
shape (334815, 14)
arz range: -10.23412 to 11.799119999999995
shape (332136, 14)
grx range: -1.0216 to 0.95294
shape (328470, 14)
gry range: -1.1437 to 0.90965
shape (323468, 14)
grz range: -0.7069 to 1.125
shape (318789, 14)


In [17]:
le=LabelEncoder()#computer dosnt understand text so it converts them into numbers 
df['Activity']=le.fit_transform(df['Activity'])#TFIFD can also be used but prefereable for more features compaitable with random forrest
df['subject']=le.fit_transform(df['subject'])

In [18]:
X=df1.drop(['Activity','subject'],axis=1)
y=df1['Activity']
X_Train,X_Test,y_Train,y_Test=train_test_split(X,y,test_size=0.25,random_state=42)



In [19]:
def resultsSummarizer(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)#tells us where the model went wrong in which part if 91percent accurate where was 9percent
    acc = accuracy_score(y_true, y_pred)

    # plt.figure(figsize=(15,15))

    # sns.heatmap(cm, annot=True, cmap="Blues", xticklabels=activity_label.values(),
    #             yticklabels=activity_label.values())

    # plt.title('Confusion Matrix')
    # plt.show()

    print('Accuracy Score: ' + '{:.4%}'.format(acc))

In [20]:
rs=RobustScaler()
X_Train_rs=rs.fit_transform(X_Train)# fit_transform is used to fit the scaler to the training data and then transform it. It calculates the median and interquartile range of the training data and uses these values to scale the training data.
X_Test_rs=rs.transform(X_Test)#this ensures machine is not seeing answers that is why we not doing fit


In [21]:
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_Train_rs,y_Train)
y_pred_knn=knn.predict(X_Test_rs)
resultsSummarizer(y_Test, y_pred_knn)

Accuracy Score: 93.9610%


In [22]:
rn=RandomForestClassifier(n_estimators=100)
rn.fit(X_Train_rs,y_Train)
y_pred_rn=rn.predict(X_Test_rs)
resultsSummarizer(y_Test, y_pred_rn)

Accuracy Score: 97.1568%


In [23]:
lr=LogisticRegression()
lr.fit(X_Train_rs,y_Train)
y_pred=lr.predict(X_Test_rs)
resultsSummarizer(y_Test, y_pred)

Accuracy Score: 60.4331%


c:\Users\sahil\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
svm=SVC(kernel='rbf',C=0.1)
svm.fit(X_Train_rs,y_Train)
y_pred_svm=svm.predict(X_Test_rs)
resultsSummarizer(y_Test, y_pred_svm)